# STAC Catalog Builder — Crop production (Zarr)

This notebook generates a STAC catalog for preprocessed **Zarr** station time-series datasets from RIBASIM HIS crop-production outputs. It follows the same publish workflow as `11_salinity.ipynb` (configure → build collection/items → save locally → upload to GCS) but uses the Zarr patterns from `01_shorelinemonitor_stacs.py` (`gen_zarr_asset`, `add_datacube`, `CoCliCoZarrLayout`).

**Prerequisites:** run `notebooks/12_crop_production.ipynb` first to produce the Zarr stores and `metadata_crop_production.json`.

## Pipeline overview

| Step | Purpose |
|------|---------|
| 1. Configure paths and options | Define input/output locations, Zarr stores, and cloud targets |
| 2. Define STAC helper functions | Reusable builders for collections, items, and Zarr assets |
| 3. Create the STAC collection | Instantiate the dataset-level catalog container from metadata JSON |
| 4. Add datacube metadata | Describe time/station dimensions and variables from the Zarr stores |
| 5. Build one test STAC item | Validate item construction on a single Zarr store |
| 6. Build all STAC items | Register one STAC item per Zarr store |
| 7. Save STAC catalog locally | Persist the collection and items to `STAC/data/current/` |
| 8. Upload Zarr stores to Google Cloud | Publish Zarr assets to GCS |
| 9. Upload STAC catalog to Google Cloud | Publish the catalog JSON to GCS |

There is no GeoServer/WMS step — these are station time-series Zarr stores, not COG rasters.

In [101]:
import datetime
import json
from pathlib import Path
from posixpath import join as urljoin

import pandas as pd
import pystac
import xarray as xr
from pystac import Summaries
from pystac.stac_io import DefaultStacIO

from coclicodata.coclico_stac.datacube import add_datacube
from coclicodata.coclico_stac.extension import CoclicoExtension, CollectionCoclicoExtension
from coclicodata.coclico_stac.layouts import CoCliCoZarrLayout
from coclicodata.coclico_stac.templates import gen_zarr_asset, get_template_collection
from coclicodata.etl.cloud_utils import (
    dataset_to_google_cloud,
    dir_to_google_cloud,
    file_to_google_cloud,
    load_google_credentials,
)
from coclicodata.etl.extract import zero_terminated_bytes_as_str

## 1) Configure paths and options

Define local and cloud paths, then load the metadata JSON. Zarr stores are discovered from `data_dir`.

Update `data_dir` and `metadata_path` if your preprocessed outputs live elsewhere.

In [102]:
repo_root = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")
stac_dir = repo_root / "global-coastal-atlas/STAC/data/current"

data_dir = Path(
    r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production"
)
metadata_path = data_dir / "metadata_crop_production.json"
google_cred_path = Path(
    r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\google_credentials.json"
)

ZARR_STORES = [
    {
        "id": "RIB_CULT_prod",
        "filename": "RIB_CULT_prod.zarr",
        "title": "Crop production",
        "description": "RIBASIM HIS crop production variables (Mcm)",
    },
    {
        "id": "RIB_ADVIR_dmnd",
        "filename": "RIB_ADVIR_dmnd.zarr",
        "title": "Crop area demand",
        "description": "RIBASIM HIS crop hectares per season",
    },
]

gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"

# Datacube / frontend settings (station time-series, no mapbox)
TEMPORAL_DIMENSION = "time"
ADDITIONAL_DIMENSIONS = ["station"]
DIMENSIONS_TO_IGNORE = ["station"]
REFERENCE_SYSTEM = "EPSG:4326"
PLOT_X_AXIS = "time"
PLOT_TYPE = "line"
PLOT_SERIES = ""

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

with open(metadata_path) as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "SPATIAL_EXTENT",
    "TEMPORAL_EXTENT",
    "COLLECTION_ID",
    "TITLE",
    "DESCRIPTION",
    "LICENSE",
    "PROVIDERS",
    "KEYWORDS",
    "UNITS",
    "MEDIA_TYPE",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(
        f"Missing required metadata keys in {metadata_path}: {', '.join(missing_keys)}"
    )

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
href_prefix = urljoin(gcs_protocol, bucket_name, bucket_proj, proj_name)

for store in ZARR_STORES:
    zarr_path = data_dir / store["filename"]
    if not zarr_path.exists():
        raise FileNotFoundError(f"Zarr store not found: {zarr_path}")

print("Zarr input:", data_dir)
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Zarr stores:", [s["filename"] for s in ZARR_STORES])
print("Temporal extent:", metadata["TEMPORAL_EXTENT"])
print("Spatial extent:", metadata["SPATIAL_EXTENT"])

Zarr input: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_production
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_production
Collection id: crop_production
Zarr stores: ['RIB_CULT_prod.zarr', 'RIB_ADVIR_dmnd.zarr']
Temporal extent: ['2014-01-01T00:00:00', '2016-12-16T00:00:00']
Spatial extent: [104.532771, 8.580031, 107.024243, 11.244372]


## 2) Define STAC helper functions

Core building blocks for catalog construction. Functions take `metadata` as an explicit argument so they can be moved to a `.py` module later.

In [103]:
def parse_license(metadata: dict) -> str:
    license_ = metadata["LICENSE"]
    if "Creative Commons" in license_ and "4.0" in license_:
        return "CC-BY-4.0"
    return license_


def create_providers(metadata: dict) -> list[pystac.Provider]:
    return [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[
                pystac.provider.ProviderRole.PRODUCER,
                pystac.provider.ProviderRole.LICENSOR,
            ],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[
                pystac.provider.ProviderRole.PROCESSOR,
                pystac.provider.ProviderRole.HOST,
            ],
            url="https://deltares.nl",
        ),
    ]


def create_extent(metadata: dict) -> pystac.Extent:
    start_datetime = datetime.datetime.strptime(
        metadata["TEMPORAL_EXTENT"][0].split("T")[0], "%Y-%m-%d"
    )
    end_datetime = None
    if len(metadata["TEMPORAL_EXTENT"]) > 1 and metadata["TEMPORAL_EXTENT"][1]:
        end_datetime = datetime.datetime.strptime(
            metadata["TEMPORAL_EXTENT"][1].split("T")[0], "%Y-%m-%d"
        )
    return pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[start_datetime, end_datetime]]),
    )


def bbox_geometry_from_extent(spatial_extent: list[float]) -> tuple[dict, list[float]]:
    west, south, east, north = spatial_extent
    bbox = [west, south, east, north]
    geometry = {
        "type": "Polygon",
        "coordinates": [
            [
                [west, south],
                [east, south],
                [east, north],
                [west, north],
                [west, south],
            ]
        ],
    }
    return geometry, bbox


def create_collection(
    metadata: dict,
    collection_id: str,
    *,
    template_fp: Path,
) -> pystac.Collection:
    """Build collection from the CoCliCo STAC template (includes Coclico extension)."""
    extent = create_extent(metadata)
    collection = get_template_collection(
        template_fp=str(template_fp),
        collection_id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        keywords=metadata["KEYWORDS"],
        license=parse_license(metadata),
        spatial_extent=[metadata["SPATIAL_EXTENT"]],
        temporal_extent=extent.temporal.intervals,
        providers=create_providers(metadata),
    )

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": metadata.get("MEDIA_TYPE", "application/zarr"),
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["DESCRIPTION"],
            "xarray:storage_options": {"token": "google_default"},
        }
    }
    return collection


def open_zarr_store(zarr_path: Path) -> xr.Dataset:
    ds = xr.open_zarr(zarr_path)
    return zero_terminated_bytes_as_str(ds)


def zarr_storage_href(store: dict, href_prefix: str) -> str:
    return urljoin(href_prefix, store["filename"])


def process_zarr_store(
    store: dict,
    zarr_path: Path,
    metadata: dict,
    href_prefix: str,
) -> tuple[pystac.Item, xr.Dataset]:
    ds = open_zarr_store(zarr_path)
    geometry, bbox = bbox_geometry_from_extent(metadata["SPATIAL_EXTENT"])
    item_datetime = pd.to_datetime(ds.time.values[0]).to_pydatetime()

    item = pystac.Item(
        id=store["id"],
        geometry=geometry,
        bbox=bbox,
        datetime=item_datetime,
        properties={"store": store["id"], "description": store["description"]},
    )
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)
    item.properties["deltares:item_key"] = store["id"]

    storage_href = zarr_storage_href(store, href_prefix)
    item.add_asset("data", gen_zarr_asset(store["title"], storage_href))
    return item, ds


def add_datacube_from_store(
    collection: pystac.Collection,
    ds: xr.Dataset,
    *,
    reference_system: str,
) -> pystac.Collection:
    return add_datacube(
        stac_obj=collection,
        ds=ds,
        temporal_dimension=TEMPORAL_DIMENSION,
        x_dimension=None,
        y_dimension=None,
        additional_dimensions=ADDITIONAL_DIMENSIONS,
        reference_system=reference_system,
    )


def build_variable_summaries(ds: xr.Dataset, label: str) -> dict:
    options = []
    for var_name in ds.data_vars:
        long_name = ds[var_name].attrs.get("long_name", str(var_name))
        options.append({"label": long_name, "value": var_name})
    return {"label": label, "options": options}


def apply_coclico_collection_props(collection: pystac.Collection, metadata: dict) -> None:
    """Set Coclico frontend props without CoclicoExtension.ext().

    pystac >= 1.9 raises ExtensionNotImplemented from CoclicoExtension.ext() because
    has_extension() does not recognise the Coclico schema URI. Instantiating
    CollectionCoclicoExtension directly is the same pattern used internally.
    """
    schema_uri = CoclicoExtension.get_schema_uri()
    if collection.stac_extensions is None:
        collection.stac_extensions = [schema_uri]
    elif not any(uri.endswith("json-schema/schema.json") for uri in collection.stac_extensions):
        collection.stac_extensions.append(schema_uri)

    coclico_ext = CollectionCoclicoExtension(collection)
    coclico_ext.units = metadata["UNITS"]
    coclico_ext.plot_series = PLOT_SERIES
    coclico_ext.plot_x_axis = PLOT_X_AXIS
    coclico_ext.plot_type = PLOT_TYPE
    coclico_ext.min_ = 0
    coclico_ext.linear_gradient = []


def verify_local_stac_outputs(stac_dir: Path, collection_id: str, *, expected_items: int) -> None:
    """Fail fast if step 7 did not write the expected STAC files."""
    collection_json = stac_dir / collection_id / "collection.json"
    item_jsons = [
        p for p in (stac_dir / collection_id).rglob("*.json") if p.name != "collection.json"
    ]
    catalog_has_child = False
    catalog_path = stac_dir / "catalog.json"
    if catalog_path.is_file():
        with open(catalog_path, encoding="utf-8") as f:
            catalog_has_child = f"./{collection_id}/collection.json" in json.dumps(json.load(f))

    missing = []
    if not collection_json.is_file():
        missing.append(str(collection_json))
    if len(item_jsons) < expected_items:
        missing.append(f"{expected_items} item JSON files (found {len(item_jsons)})")
    if not catalog_has_child:
        missing.append(f"catalog child link ./{collection_id}/collection.json")

    if missing:
        raise FileNotFoundError(
            "STAC outputs incomplete after save. Re-run steps 3–6 in the same kernel, "
            "then step 7 again.\nMissing: " + "; ".join(missing)
        )

    print(f"Verified: {collection_json}")
    print(f"Verified: {len(item_jsons)} item JSON file(s)")
    print(f"Verified: catalog.json links to ./{collection_id}/collection.json")

## 3) Create the STAC collection

Instantiate the STAC collection from metadata JSON. This is independent of individual Zarr stores.

In [104]:
template_fp = stac_dir / "template" / "collection.json"
collection = create_collection(metadata, collection_id, template_fp=template_fp)
layout = CoCliCoZarrLayout()

print(f"Collection ready: {collection.id}")
print(f"Title: {metadata['TITLE']}")
print(f"Provider: {metadata['PROVIDERS']['name']}")
print(f"Media type: {metadata['MEDIA_TYPE']}")
print(f"Coclico extensions: {collection.stac_extensions}")

Collection ready: crop_production
Title: Title
Provider: Provider Name
Media type: application/zarr
Coclico extensions: ['https://raw.githubusercontent.com/openearth/coclicodata/feat/update-deltares-stac-properties/json-schema/schema.json', 'https://stac-extensions.github.io/item-assets/v1.0.0/schema.json']


## 4) Add datacube metadata

Each Zarr store stays independent. The collection datacube describes the production store only (time + station dimensions). Each store also gets its own collection asset and variable summary keyed by store id.

In [105]:
store_datasets = {
    store["id"]: open_zarr_store(data_dir / store["filename"]) for store in ZARR_STORES
}

# Datacube metadata for the production store only (stores are not combined).
collection = add_datacube_from_store(
    collection,
    store_datasets[ZARR_STORES[0]["id"]],
    reference_system=REFERENCE_SYSTEM,
)

for store in ZARR_STORES:
    collection.add_asset(
        store["id"],
        gen_zarr_asset(store["title"], zarr_storage_href(store, href_prefix)),
    )

collection.summaries = Summaries({})
for store in ZARR_STORES:
    collection.summaries.add(
        store["id"],
        build_variable_summaries(store_datasets[store["id"]], store["title"]),
    )

apply_coclico_collection_props(collection, metadata)

print("cube:dimensions:", list(collection.extra_fields.get("cube:dimensions", {}).keys()))
print("cube:variables:", len(collection.extra_fields.get("cube:variables", {})))
print("collection assets:", list(collection.assets.keys()))
print("summaries:", list(collection.summaries.to_dict().keys()))

cube:dimensions: ['time', 'station']
cube:variables: 23
collection assets: ['RIB_CULT_prod', 'RIB_ADVIR_dmnd']
summaries: ['RIB_CULT_prod', 'RIB_ADVIR_dmnd']


## 5) Build one test STAC item

Validate item construction on a single Zarr store before batch processing.

In [106]:
test_store = ZARR_STORES[0]
test_item, _ = process_zarr_store(
    test_store,
    data_dir / test_store["filename"],
    metadata,
    href_prefix,
)

print(f"Test item id: {test_item.id}")
print(f"Test item datetime: {test_item.datetime}")
print(f"Data asset href: {test_item.assets['data'].href}")

Test item id: RIB_CULT_prod
Test item datetime: 2014-01-01 00:00:00
Data asset href: https://storage.googleapis.com/gca-data-public/gca/crop_production/RIB_CULT_prod.zarr


## 6) Build all STAC items

Generate one STAC item per Zarr store and register them with the collection.

In [107]:
items = []
item_rows = []
item_errors = []

for store in ZARR_STORES:
    zarr_path = data_dir / store["filename"]
    try:
        item, _ = process_zarr_store(store, zarr_path, metadata, href_prefix)
        item_href = stac_dir / collection_id / store["id"] / f"{store['id']}.json"
        item.set_self_href(str(item_href))
        collection.add_item(item, strategy=layout)
        items.append(item)
        item_rows.append({"store": store["id"], "item_id": item.id, "stac_href": str(item_href)})
        print(f"Created item: {item.id}")
    except Exception as exc:
        item_errors.append({"store": store["id"], "error": str(exc)})

print(f"Items created: {len(item_rows)}")
print(f"Errors: {len(item_errors)}")
if item_errors:
    for row in item_errors:
        print(row)

Created item: RIB_CULT_prod
Created item: RIB_ADVIR_dmnd
Items created: 2
Errors: 0


## 7) Save STAC catalog locally

Same pattern as `11_salinity.ipynb`: serialize the collection and items, merge into the root `catalog.json`, and save the self-contained catalog under `STAC/data/current/`.

**Expected git changes after this step**

| Path | Should change? | Why |
|------|----------------|-----|
| `crop_production/` | Yes | New/updated collection and items |
| `catalog.json` | Yes | New child link to `crop_production` |
| Other `*/collection.json` | No* | *Should be identical after round-trip; see note below |

**If another collection JSON does change**, it is usually because pystac reloads every catalog child into memory and writes it back. Fields that are not fully preserved on load (e.g. some `stac_extensions` entries on collections that were not rebuilt in this notebook) can be dropped even though the dataset content is unchanged. That is a pystac round-trip limitation, not a change to the underlying data. Use `DefaultStacIO()` (as in salinity) so formatting stays consistent.

In [108]:
stac_io = DefaultStacIO()

for store in ZARR_STORES:
    (stac_dir / collection_id / store["id"]).mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
    dest_href=str(stac_dir),
    stac_io=stac_io,
)

collection.validate_all()
catalog.validate_all()
verify_local_stac_outputs(stac_dir, collection_id, expected_items=len(ZARR_STORES))
print(f"STAC saved to: {stac_dir}")

Removed existing child: crop_production
Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_production\collection.json
Verified: 2 item JSON file(s)
Verified: catalog.json links to ./crop_production/collection.json
STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 8) Upload Zarr stores to Google Cloud (optional)

Publish each Zarr store to `gs://{bucket_name}/{bucket_proj}/{proj_name}/{filename}` using `dataset_to_google_cloud`, matching the hrefs referenced in STAC assets.

Requires valid Google Cloud credentials at `google_cred_path`. Skip if Zarr stores are already on GCS.

In [109]:
load_google_credentials(google_token_fp=google_cred_path)

for store in ZARR_STORES:
    zarr_path = data_dir / store["filename"]
    print(f"Uploading {store['filename']}...")
    dataset_to_google_cloud(
        ds=zarr_path,
        gcs_project=gcs_project,
        bucket_name=bucket_name,
        bucket_proj=bucket_proj,
        zarr_filename=urljoin(proj_name, store["filename"]),
    )
    print(f"Done: {store['id']}")

Google Application Credentials load into environment.
Uploading RIB_CULT_prod.zarr...
Writing to zarr store at gca-data-public/gca/crop_production/RIB_CULT_prod.zarr...


C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:230: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:243: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Done!
Done: RIB_CULT_prod
Uploading RIB_ADVIR_dmnd.zarr...
Writing to zarr store at gca-data-public/gca/crop_production/RIB_ADVIR_dmnd.zarr...
Done!
Done: RIB_ADVIR_dmnd


## 9) Upload STAC catalog to Google Cloud (optional)

Publish **only** the new `crop_production` STAC folder and the updated root `catalog.json` to `gs://gca-data-public/gca/gca-stac-7/`. This is enough for the collection to appear in the bucket without re-uploading every other dataset.

**Prerequisite:** step 7 must pass `verify_local_stac_outputs` first.

In [110]:
verify_local_stac_outputs(stac_dir, collection_id, expected_items=len(ZARR_STORES))

load_google_credentials(google_token_fp=google_cred_path)

# 1) Upload crop_production STAC folder -> gs://.../gca-stac-7/crop_production/
dir_to_google_cloud(
    dir_path=str(stac_dir / collection_id),
    gcs_project=gcs_project,
    bucket_name=bucket_name,
    bucket_proj=bucket_proj,
    dir_name=urljoin(stac_cloud_name, collection_id),
)

# 2) Upload root catalog.json (contains the crop_production child link)
file_to_google_cloud(
    file_path=str(stac_dir / "catalog.json"),
    gcs_project=gcs_project,
    bucket_name=bucket_name,
    bucket_proj=bucket_proj,
    dir_name=stac_cloud_name,
    file_name="catalog.json",
)

print(f"Uploaded STAC collection: gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/{collection_id}/")
print(f"Uploaded catalog: gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/catalog.json")

Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_production\collection.json
Verified: 2 item JSON file(s)
Verified: catalog.json links to ./crop_production/collection.json
Google Application Credentials load into environment.
Writing to directory at gca-data-public/gca/gca-stac-7/crop_production...


C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:230: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:243: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Done!
Uploaded STAC collection: gs://gca-data-public/gca/gca-stac-7/crop_production/
Uploaded catalog: gs://gca-data-public/gca/gca-stac-7/catalog.json
